# ESMFold2 — CYP3A4 pose generation
(c) Byron DeLaBarre / The Consulting Biochemist 2026

Rebuilt from `esmfold2_cyp3a4_probe_lean.ipynb`. Three changes drove the
refactor, and one consequence of them is easy to miss.

**1. Stereochemistry.** SMILES come from `CHALLENGE_CSV`, optionally overridden
per compound in cell 5. Every compound is then audited with RDKit and the cell
**raises** if any stereocentre is still unassigned — the input can no longer be
ambiguous without you being told. A compound may also carry *two* SMILES (both
enantiomers), in which case they fold as `sample_1` and `sample_2`: the
downstream suite already keys on `(cmpd, seed, sample)`, so OrbMol ΔE and
PoseBusters decide between them instead of you guessing.

**2. Sequence 21–503.** Cell 4 derives the numbering of whatever sequence the
template JSON holds (by matching pocket residue identities, not by assumption),
slices it to the crystallography construct, and sets `SEQ_FIRST = 21`. The MSA
is sliced to the same columns — a query row that no longer matches the sequence
is the kind of thing ESMFold2 accepts and then folds badly.

**3. More subpockets.** Cell 6 clusters the pocket into `N_SUBPOCKETS`,
optionally adds unions of adjacent pairs, and always adds the whole pocket. It
prints the fold count before anything is billed.

**The consequence:** truncating the N-terminus shifts every output residue
number by `SEQ_FIRST - 1`. Cys442 would come back as residue 422. `RENUMBER_OUT`
(default on) writes the predicted cif back in native P08684 numbering so every
artifact stays comparable with 1TQN, with your earlier runs, and with the
pocket definitions the refinement and scoring notebooks use.

Stage order: cells 2–8 set up and are cheap; **cell 9 is the only cell that
spends tokens.**

In [ ]:
# ============================================================================
# 1 -- imports and client
# ============================================================================
# !pip install esm --ignore-requires-python     # needs Python >=3.12,<3.13

import os, json, time, re
from pathlib import Path
from getpass import getpass
import numpy as np, pandas as pd, gemmi

from esm.sdk import esmfold2_client
from esm.sdk.api import FoldingConfig, ESMProteinError
from esm.utils.structure import input_builder
from esm.utils.msa import MSA

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

TOKEN = os.getenv("BIOHUB_KEY") or getpass("Biohub token: ")
MODEL = "esmfold2-2026-05"
client = esmfold2_client(model=MODEL, url="https://biohub.ai", token=TOKEN)
print(f"client ready: {MODEL}")

_AA1 = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
        "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F",
        "PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V"}
_AA3 = {v: k for k, v in _AA1.items()}

In [ ]:
# ============================================================================
# 2 -- CONFIG. Every knob in this notebook lives here.
# ============================================================================
CYP_PREDICTIONS      = Path(os.getenv("CYP_PREDICTIONS", "."))
CYP_BASE      = Path(os.getenv("CYP_BASE", "."))
CHALLENGE_CSV = Path(os.getenv("CHALLENGE_CSV"))
TEMPLATE_JSON = CYP_PREDICTIONS / "openadmet20_query_cyp3A4.json"   # source of seq+MSA only
ALIGN_DIR     = CYP_PREDICTIONS / "alignments"

RUN_NAME = "esmfold2_cyp3a4_run11"
RUN_DIR  = CYP_PREDICTIONS / RUN_NAME
CIF_DIR  = RUN_DIR / "all_predictions"
SCORES_CSV = CIF_DIR / "harvest_all_scores.csv"

PROT_CHAIN, HEME_CHAIN, LIG_CHAIN = "A", "B", "C"
HEME_CCD = ["HEM"]

# --- (2) crystallography construct ----------------------------------------
# P08684 numbering. 21-503 = 483 residues; the Delta3-22 crystal constructs
# (1TQN and relatives) are ordered from about here.
SEQ_FIRST, SEQ_LAST = 1, 503
RENUMBER_OUT = True        # write cifs back in native numbering -- see cell 1

# --- MSA -------------------------------------------------------------------
USE_MSA      = True
MSA_MAX_SEQS = 1000
MSA_MIN_COVERAGE = 0.30    # drop rows that are mostly gaps after slicing

# --- (3) subpocket conditioning -------------------------------------------
POCKET_SET      = "full"   # "full" (46 res) | "nonbinder" (16 res)
N_SUBPOCKETS    = 5        # cell 6 sweeps this and prints the cost first
MIN_CLUSTER     = 3        # clusters smaller than this are merged into the nearest
LINKAGE         = "ward"   # ward gives balanced clusters; average collapses to
                           # one giant cluster plus singletons at N>6
ADD_PAIR_SETS   = 2        # unions of the N closest subpocket pairs (0 = none)
ADD_WHOLE_POCKET = True    # one set conditioned on every verified residue
ADD_HEME_CONTACT = False   # True puts (HEME_CHAIN, 0) in every set

# Reuse a previous run's subpockets verbatim instead of re-clustering. Set to
# None to cluster as usual. Residues are still verified against the sequence
# and the reference structure; only the GROUPING is taken as given, so a
# change of LINKAGE, MIN_CLUSTER or N_SUBPOCKETS cannot silently alter it.
# Lift these straight out of an old run_definition.json["subpockets"].
EXPLICIT_SUBPOCKETS = {
    1: [76, 214, 215, 216, 217, 218, 219, 220, 221, 224],
    2: [239, 240, 241],
    3: [105, 106, 107, 108, 109, 114, 119, 120, 301],
    4: [208, 209, 210, 211, 212, 213, 304, 305, 306, 307, 308, 309],
    5: [53, 57, 369, 370, 371, 372, 373, 374, 480, 481, 482, 484],
    #6: [119, 304, 305, 306, 369, 370, 372, 373],   # NEW1, catalytic site
}

# --- compounds -------------------------------------------------------------
ONLY_COMPOUNDS = ()        # () = all 20; else e.g. ("cmpd_0", "cmpd_7")

# Reference structure for pocket CA coordinates. Must be in native P08684
# numbering. Now that the construct matches the crystal form, 1TQN is the
# natural choice; any of your own predictions works too.
REF_PDB = Path(os.getenv("CYP_REF_PDB", "")) if os.getenv("CYP_REF_PDB") else None

CONFIG = FoldingConfig(num_loops=10, num_sampling_steps=100,
                       include_pae=True, include_pair_chains_iptm=True)

CIF_DIR.mkdir(parents=True, exist_ok=True)
print(f"run      {RUN_NAME}  -> {CIF_DIR}")
print(f"sequence {SEQ_FIRST}-{SEQ_LAST} ({SEQ_LAST - SEQ_FIRST + 1} residues)"
      f"   renumber output: {RENUMBER_OUT}")
print(f"template json exists: {TEMPLATE_JSON.is_file()}   "
      f"challenge csv: {CHALLENGE_CSV.is_file()}")

In [ ]:
# ============================================================================
# 3 -- POCKET definitions, in native P08684 / 1TQN numbering.
# ============================================================================
# 46 active-site residues, defined against all ligands
POCKET_FULL = {
     53: "TYR",  57: "PHE",  76: "ASP", 105: "ARG", 106: "ARG", 107: "PRO",
    108: "PHE", 109: "GLY", 114: "MET", 119: "SER", 120: "ILE", 208: "LYS",
    209: "LYS", 210: "LEU", 211: "LEU", 212: "ARG", 213: "PHE", 214: "ASP",
    215: "PHE", 216: "LEU", 217: "ASP", 218: "PRO", 219: "PHE", 220: "PHE",
    221: "LEU", 224: "THR", 239: "CYS", 240: "VAL", 241: "PHE", 301: "ILE",
    304: "PHE", 305: "ALA", 306: "GLY", 307: "TYR", 308: "GLU", 309: "THR",
    369: "ILE", 370: "ALA", 371: "MET", 372: "ARG", 373: "LEU", 374: "GLU",
    480: "GLY", 481: "GLY", 482: "LEU", 484: "GLN",
}
# 16 residues, derived from the non-binder subset
POCKET_NONBINDER = {
     57: "PHE",  76: "ASP", 105: "ARG", 106: "ARG", 119: "SER", 223: "ILE",
    224: "THR", 227: "PRO", 230: "ILE", 301: "ILE", 305: "ALA", 309: "THR",
    370: "ALA", 372: "ARG", 374: "GLU", 482: "LEU",
}
POCKET = {"full": POCKET_FULL, "nonbinder": POCKET_NONBINDER}[POCKET_SET]
CYS_RESNUM = 442          # proximal Cys, native numbering

print(f"pocket '{POCKET_SET}': {len(POCKET)} residues "
      f"{min(POCKET)}-{max(POCKET)}")

POCKET_SET       = "full"
ADD_PAIR_SETS    = 0
ADD_WHOLE_POCKET = False
EXPLICIT_SUBPOCKETS = {
    1: [76, 214, 215, 216, 217, 218, 219, 220, 221, 224],
    2: [239, 240, 241],
    3: [105, 106, 107, 108, 109, 114, 119, 120, 301],
    4: [208, 209, 210, 211, 212, 213, 304, 305, 306, 307, 308, 309],
    5: [53, 57, 369, 370, 371, 372, 373, 374, 480, 481, 482, 484],
    #6: [119, 304, 305, 306, 369, 370, 372, 373]   # NEW1, catalytic site
}

In [ ]:
# ============================================================================
# 4 -- SEQUENCE: load, derive its numbering, slice to SEQ_FIRST..SEQ_LAST,
#      and slice the MSA to the same columns.
#
# The offset is DERIVED, not assumed: the correct one is the only value that
# reproduces all of the pocket residue identities.
# ============================================================================
_q = json.loads(TEMPLATE_JSON.read_text())
_queries = _q["queries"]
_entries = list(_queries.values()) if isinstance(_queries, dict) else _queries

prot = None
for ent in _entries:
    chains = ent if isinstance(ent, list) else ent.get("chains", ent)
    if isinstance(chains, dict):
        chains = [chains]
    for ch in chains:
        if isinstance(ch, dict) and str(ch.get("molecule_type", "")).lower() \
                in ("protein", "polypeptide", "polypeptide(l)"):
            prot = ch; break
    if prot:
        break
if prot is None:
    raise RuntimeError("no protein chain in the template json; inspect _entries")

FULL_SEQ   = prot["sequence"]
_json_msa  = prot.get("main_msa_file_paths") or []
if isinstance(_json_msa, str):
    _json_msa = [_json_msa]
print(f"template sequence: {len(FULL_SEQ)} residues  "
      f"{FULL_SEQ[:10]}...{FULL_SEQ[-10:]}")


def derive_first_resnum(seq, pocket):
    """The residue number of seq[0], from pocket residue identities alone."""
    best = []
    for off in range(-30, 131):
        hit = tot = 0
        for n, nm3 in pocket.items():
            i = n - off
            if 0 <= i < len(seq):
                tot += 1
                hit += seq[i] == _AA1.get(nm3, "?")
        if tot:
            best.append((hit / tot, hit, tot, off))
    best.sort(reverse=True)
    return best[0], best[:4]


(_frac, _hit, _tot, FULL_FIRST), _top = derive_first_resnum(FULL_SEQ, POCKET)
print(f"\nderived numbering of the template sequence:")
for f, h, t, o in _top:
    print(f"   first_resnum={o:4d}  {h:2d}/{t:2d} pocket residues match"
          + ("   <-- using this" if o == FULL_FIRST else ""))
if _frac < 1.0:
    raise RuntimeError(
        f"best offset reproduces only {_hit}/{_tot} pocket residues. The "
        f"template sequence is not the construct POCKET was defined against.")

# --- slice to the crystallography construct -------------------------------
i0 = SEQ_FIRST - FULL_FIRST
i1 = SEQ_LAST  - FULL_FIRST + 1
if i0 < 0 or i1 > len(FULL_SEQ):
    raise ValueError(
        f"cannot slice {SEQ_FIRST}-{SEQ_LAST} out of a sequence numbered "
        f"{FULL_FIRST}-{FULL_FIRST + len(FULL_SEQ) - 1}")
PROT_SEQ  = FULL_SEQ[i0:i1]
SEQ_SHIFT = SEQ_FIRST - 1          # output resnum + SEQ_SHIFT = native resnum
print(f"\nsliced [{i0}:{i1}] -> {len(PROT_SEQ)} residues, "
      f"native {SEQ_FIRST}-{SEQ_LAST}")
print(f"  starts {PROT_SEQ[:15]}...")
print(f"  ends   ...{PROT_SEQ[-15:]}")
assert len(PROT_SEQ) == SEQ_LAST - SEQ_FIRST + 1


def res_index(resnum, expect=None):
    """Native residue number -> 0-based index into PROT_SEQ (what
    PocketConditioning contacts want)."""
    i = resnum - SEQ_FIRST
    if not 0 <= i < len(PROT_SEQ):
        raise ValueError(f"residue {resnum} is outside {SEQ_FIRST}-{SEQ_LAST}")
    if expect and PROT_SEQ[i] != expect:
        raise ValueError(f"residue {resnum} is {PROT_SEQ[i]!r}, expected "
                         f"{expect!r} -- numbering is wrong")
    return i


# --- every pocket residue must still verify after the slice ---------------
_bad = []
for n, nm3 in sorted(POCKET.items()):
    try:
        res_index(n, expect=_AA1[nm3])
    except ValueError as ex:
        _bad.append(f"{nm3}{n}: {ex}")
print(f"\n{len(POCKET) - len(_bad)}/{len(POCKET)} pocket residues verified "
      f"against the sliced sequence")
for b in _bad:
    print(f"  [!] {b}")
if _bad:
    raise RuntimeError("pocket residues fall outside the construct -- widen "
                       "SEQ_FIRST/SEQ_LAST or fix POCKET")

print(f"Cys{CYS_RESNUM} -> index {res_index(CYS_RESNUM, expect='C')}  "
      f"(output residue {CYS_RESNUM - SEQ_SHIFT} before renumbering)")

# --- MSA, sliced to the same columns --------------------------------------
FULL_MSA = None
if USE_MSA:
    cand = [Path(p) for p in _json_msa if str(p).endswith(".a3m")]
    cand += sorted(ALIGN_DIR.glob("*.a3m")) if ALIGN_DIR.is_dir() else []
    cand = [p for p in cand if p.is_file()]
    if not cand:
        print("\n[!] no .a3m found -- folding single-sequence, which is much "
              "worse for complexes")
    else:
        _raw = MSA.from_a3m(path=str(cand[0]), remove_insertions=True,
                            max_sequences=MSA_MAX_SEQS)
        print(f"\nMSA {cand[0].name}: depth {_raw.depth}, length {_raw.seqlen}")
        if _raw.sequences[0] != FULL_SEQ:
            raise RuntimeError(
                "the MSA query row does not match the template sequence, so "
                "slicing it by column would misalign every row")
        # remove_insertions aligns every row to the query, so column i IS
        # query position i and the same slice applies to all rows
        _sliced = [s[i0:i1] for s in _raw.sequences]
        _keep = [s for s in _sliced
                 if (len(s) - s.count("-")) / len(s) >= MSA_MIN_COVERAGE]
        if _keep[0] != PROT_SEQ:
            raise RuntimeError("sliced MSA query row != sliced sequence")
        FULL_MSA = MSA.from_sequences(_keep)
        print(f"  sliced -> length {len(_keep[0])}, depth {len(_keep)} "
              f"({_raw.depth - len(_keep)} rows dropped below "
              f"{MSA_MIN_COVERAGE:.0%} coverage)")
        print(f"  query row matches sliced sequence: {_keep[0] == PROT_SEQ}")

In [ ]:
# ============================================================================
# 5 -- SMILES and the stereochemistry audit.
#
# Order: CHALLENGE_CSV, then STEREO_OVERRIDE on top. The audit then RAISES if
# any compound still has an unassigned stereocentre, so an ambiguous input
# cannot reach the folding cell silently.
#
# A value may be a LIST of SMILES -- both enantiomers, say. Each folds as its
# own `sample` index, and the downstream suite ranks them against each other
# on OrbMol dE and PoseBusters instead of you having to choose.
# ============================================================================
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

# --- the three compounds OpenADMET left unspecified ------------------------
# Both configurations, verified to differ only in stereochemistry (same
# constitution, no explicit H nodes). Uncomment the one you were given, or
# give both as a list to fold both.
#
# cmpd_7  / OCNT-2395415
#   (S) "Cc1cccc(C(=O)N[C@@H](C)c2cccnc2)c1"
#   (R) "Cc1cccc(C(=O)N[C@H](C)c2cccnc2)c1"
# cmpd_10 / OCNT-2312554
#   (S) "O=C(NC[C@H]1CCc2ccccc21)c1ccnc(-n2ccnc2)c1"
#   (R) "O=C(NC[C@@H]1CCc2ccccc21)c1ccnc(-n2ccnc2)c1"
# cmpd_19 / OCNT-2312966
#   (S) "O=C1Cc2cc(CCCC(=O)N3CC[C@H]3c3ccc(F)c(F)c3)ccc2N1"
#   (R) "O=C1Cc2cc(CCCC(=O)N3CC[C@@H]3c3ccc(F)c(F)c3)ccc2N1"

STEREO_OVERRIDE = {
    # "cmpd_7":  "Cc1cccc(C(=O)N[C@@H](C)c2cccnc2)c1",
    # "cmpd_10": "O=C(NC[C@H]1CCc2ccccc21)c1ccnc(-n2ccnc2)c1",
    # "cmpd_19": ["O=C1Cc2cc(CCCC(=O)N3CC[C@H]3c3ccc(F)c(F)c3)ccc2N1",
    #             "O=C1Cc2cc(CCCC(=O)N3CC[C@@H]3c3ccc(F)c(F)c3)ccc2N1"],
}
REQUIRE_FULL_STEREO = True       # False only to deliberately fold a racemate

_ch  = pd.read_csv(CHALLENGE_CSV)
_key = _ch["id2"].astype(str).str.strip()
SMILES  = {k: [v] for k, v in zip(_key, _ch["SMILES"])}
OCNT_ID = dict(zip(_key, _ch["id"].astype(str).str.strip()))

for c, v in STEREO_OVERRIDE.items():
    if c not in SMILES:
        raise KeyError(f"STEREO_OVERRIDE: {c} is not in {CHALLENGE_CSV.name}")
    SMILES[c] = list(v) if isinstance(v, (list, tuple)) else [v]

# --- audit -----------------------------------------------------------------
rows, problems = [], []
for c in sorted(SMILES, key=lambda s: int(s.split("_")[1])):
    for vi, smi in enumerate(SMILES[c], 1):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            problems.append(f"{c} variant {vi}: SMILES does not parse"); continue
        centres = Chem.FindMolChiralCenters(
            m, includeUnassigned=True, useLegacyImplementation=False)
        unassigned = [i for i, t in centres if t == "?"]
        rows.append({"cmpd": c, "ocnt": OCNT_ID.get(c, ""), "variant": vi,
                     "n_variants": len(SMILES[c]), "heavy": m.GetNumAtoms(),
                     "centres": len(centres), "unassigned": len(unassigned),
                     "overridden": c in STEREO_OVERRIDE})
        if unassigned:
            problems.append(f"{c} variant {vi} ({OCNT_ID.get(c,'')}): "
                            f"{len(unassigned)} unassigned stereocentre(s) at "
                            f"atom(s) {unassigned}  {smi}")
    # both variants must be the same molecule apart from stereochemistry
    if len(SMILES[c]) > 1:
        flat = {Chem.MolToSmiles(Chem.MolFromSmiles(s), isomericSmiles=False)
                for s in SMILES[c]}
        if len(flat) != 1:
            problems.append(f"{c}: variants differ in constitution, not just "
                            f"stereochemistry")
        if len({Chem.MolToSmiles(Chem.MolFromSmiles(s)) for s in SMILES[c]}) \
                != len(SMILES[c]):
            problems.append(f"{c}: duplicate variants")

AUDIT = pd.DataFrame(rows)
print(AUDIT.to_string(index=False))
_nv = AUDIT.groupby("cmpd")["variant"].max()
print(f"\n{AUDIT.cmpd.nunique()} compounds, {len(AUDIT)} SMILES "
      f"({int((_nv > 1).sum())} with multiple variants), "
      f"{int(AUDIT.centres.sum())} stereocentres, "
      f"{int(AUDIT.unassigned.sum())} unassigned")

if problems:
    print("\n".join(["", "PROBLEMS:"] + [f"  [!] {p}" for p in problems]))
    if REQUIRE_FULL_STEREO:
        raise RuntimeError(
            f"{len(problems)} compound(s) still ambiguous. Fill in "
            f"STEREO_OVERRIDE (candidates are in the comment above), update "
            f"{CHALLENGE_CSV.name}, or set REQUIRE_FULL_STEREO = False.")
else:
    print("\nOK: every stereocentre in every variant is specified.")

CMPDS = [c for c in sorted(SMILES, key=lambda s: int(s.split("_")[1]))
         if not ONLY_COMPOUNDS or c in ONLY_COMPOUNDS]
print(f"folding {len(CMPDS)} compounds: {CMPDS}")

In [ ]:
# ============================================================================
# 6 -- SUBPOCKETS -> contact sets. Costs nothing; prints what cell 9 will.
#
# Clustering is on CA coordinates taken from the SEQUENCE's own numbering via
# a reference structure, so no second offset is involved: residues are looked
# up by native number and verified by identity.
# ============================================================================
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

if REF_PDB is None or not REF_PDB.is_file():
    raise FileNotFoundError(
        "set CYP_REF_PDB to a structure in native P08684 numbering "
        "(1TQN.pdb, or one of your own predictions)")

_st = gemmi.read_structure(str(REF_PDB)); _st.setup_entities()
_ref = {r.seqid.num: r for ch in _st[0] if ch.name == PROT_CHAIN for r in ch}
print(f"reference {REF_PDB.name}: chain {PROT_CHAIN}, "
      f"{len(_ref)} residues numbered {min(_ref)}-{max(_ref)}")

ca, keep, bad = [], [], []
for n in sorted(POCKET):
    want3 = POCKET[n]
    r = _ref.get(n)
    if r is None:
        bad.append((n, want3, "absent from REF_PDB")); continue
    if r.name.strip().upper() != want3:
        bad.append((n, want3, f"REF_PDB has {r.name.strip()}")); continue
    a = next((x for x in r if x.name.strip() == "CA"), None)
    if a is None:
        bad.append((n, want3, "no CA")); continue
    try:
        res_index(n, expect=_AA1[want3])
    except ValueError as ex:
        bad.append((n, want3, str(ex))); continue
    ca.append(a.pos.tolist()); keep.append(n)
ca = np.array(ca)
print(f"{len(keep)}/{len(POCKET)} pocket residues verified")
for n, w, why in bad:
    print(f"  [!] {w}{n}: {why}")
if len(keep) < len(POCKET) * 0.9:
    raise RuntimeError("too many pocket residues failed verification")

# --- cluster, then absorb clusters below MIN_CLUSTER into the nearest ------
_Z = linkage(pdist(ca), LINKAGE)


def _cluster(n_req):
    lab = fcluster(_Z, min(n_req, len(keep)), "maxclust")
    groups = {k: [i for i, l in enumerate(lab) if l == k]
              for k in sorted(set(lab))}
    while True:
        small = [k for k, v in groups.items() if len(v) < MIN_CLUSTER]
        if not small or len(groups) <= 2:
            break
        k = min(small, key=lambda k: len(groups[k]))
        cen = {j: ca[v].mean(0) for j, v in groups.items() if j != k}
        tgt = min(cen,
                  key=lambda j: np.linalg.norm(cen[j] - ca[groups[k]].mean(0)))
        groups[tgt] += groups.pop(k)
    return {i: [keep[j] for j in sorted(v)]
            for i, (_k, v) in enumerate(sorted(groups.items()), 1)}


if EXPLICIT_SUBPOCKETS:
    # --- reuse a previous run's grouping verbatim --------------------------
    subpockets = {int(k): sorted(int(r) for r in v)
                  for k, v in EXPLICIT_SUBPOCKETS.items()}
    flat = [r for v in subpockets.values() for r in v]
    dup = sorted({r for r in flat if flat.count(r) > 1})
    missing = sorted(set(flat) - set(keep))
    unused = sorted(set(keep) - set(flat))
    print(f"\nEXPLICIT_SUBPOCKETS: {len(subpockets)} sets, "
          f"{len(flat)} residues ({len(set(flat))} unique)")
    if dup:
        print(f"  [i] residues in more than one set: {dup}")
    if unused:
        print(f"  [i] verified pocket residues not used by any set: {unused}")
    if missing:
        raise KeyError(
            f"EXPLICIT_SUBPOCKETS names residues that failed verification "
            f"or are not in POCKET_SET={POCKET_SET!r}: {missing}. Check "
            f"POCKET_SET, SEQ_FIRST/SEQ_LAST and REF_PDB.")
    print("  (clustering skipped: LINKAGE / MIN_CLUSTER / N_SUBPOCKETS "
          "are not used)")
else:
    # --- how many subpockets can this pocket actually support? -------------
    # "Maximize" has a ceiling: past some N the clusters stop being
    # subpockets and become individual residues, which condition nothing.
    # Pick N off this table.
    print(f"\n--- subpocket sweep ({LINKAGE} linkage, {len(keep)} residues, "
          f"min size {MIN_CLUSTER}) ---")
    print(f"{'asked':>5} {'got':>4} {'smallest':>9}  sizes")
    for _n in range(3, min(17, len(keep))):
        _sp = _cluster(_n)
        _sz = sorted(len(v) for v in _sp.values())
        _mark = "  <-- N_SUBPOCKETS" if _n == N_SUBPOCKETS else ""
        print(f"{_n:>5} {len(_sp):>4} {min(_sz):>9}  {_sz}{_mark}")

    subpockets = _cluster(N_SUBPOCKETS)
    print(f"\n{len(subpockets)} subpockets (asked for {N_SUBPOCKETS}, "
          f"min size {MIN_CLUSTER}):")

for k, v in subpockets.items():
    print(f"  sp{k}: {len(v):2d} residues {v}")

# --- contact sets ----------------------------------------------------------
def _contacts(resnums):
    cs = [(PROT_CHAIN, res_index(n)) for n in resnums]
    return cs + [(HEME_CHAIN, 0)] if ADD_HEME_CONTACT else cs

CONTACT_SETS, SET_LABEL, SET_RESNUMS = {}, {}, {}
for k, v in subpockets.items():
    CONTACT_SETS[k] = _contacts(v); SET_LABEL[k] = f"sp{k}"; SET_RESNUMS[k] = v

if ADD_PAIR_SETS:
    cen = {k: ca[[keep.index(n) for n in v if n in keep]].mean(0)
           for k, v in subpockets.items()}
    pairs = sorted(((float(np.linalg.norm(cen[a] - cen[b])), a, b)
                    for a in subpockets for b in subpockets if a < b)
                   )[:ADD_PAIR_SETS]
    for _d, a, b in pairs:
        k = max(CONTACT_SETS) + 1
        v = sorted(subpockets[a] + subpockets[b])
        CONTACT_SETS[k] = _contacts(v)
        SET_LABEL[k] = f"sp{a}+{b}"; SET_RESNUMS[k] = v
        print(f"  sp{a}+{b} (centroids {_d:.1f} A apart): {len(v)} residues")

if ADD_WHOLE_POCKET:
    k = max(CONTACT_SETS) + 1 if CONTACT_SETS else 1
    CONTACT_SETS[k] = _contacts(keep)
    SET_LABEL[k] = "all"; SET_RESNUMS[k] = list(keep)
    print(f"  all: {len(keep)} residues (whole pocket)")

SET_IDS = sorted(CONTACT_SETS)
N_VARIANTS = sum(len(SMILES[c]) for c in CMPDS)
N_FOLDS = len(SET_IDS) * N_VARIANTS
print(f"\n{len(SET_IDS)} contact sets x {N_VARIANTS} compound-variants "
      f"= {N_FOLDS} folds")
print("  " + "  ".join(f"{SET_LABEL[k]}({len(SET_RESNUMS[k])})" for k in SET_IDS))
if N_FOLDS > 150:
    print(f"\n[!] {N_FOLDS} billed calls. Lower N_SUBPOCKETS/ADD_PAIR_SETS, or "
          f"narrow ONLY_COMPOUNDS, if that is more than you want to spend.")

In [ ]:
# 7 -- helpers. Defined once.
# ============================================================================
def _f(x):
    """Scalar from a tensor, array or number; nan if it cannot be reduced."""
    try:
        return float(x.mean().item() if hasattr(x, "mean") else x)
    except Exception:
        return float("nan")


def fold_once(fi, config, model_name=None):
    """fold_all_atom returns a result, a LIST of results, or an ESMProteinError
    that it does NOT raise. Normalise all three."""
    r = client.fold_all_atom(fi, model_name=model_name, config=config)
    if isinstance(r, ESMProteinError):
        raise RuntimeError(f"ESMProteinError: {r}")
    return r[0] if isinstance(r, list) else r


def build_input(smiles, contacts, msa_obj=None):
    return input_builder.StructurePredictionInput(
        sequences=[
            input_builder.ProteinInput(id=PROT_CHAIN, sequence=PROT_SEQ,
                                       msa=msa_obj),
            input_builder.LigandInput(id=HEME_CHAIN, ccd=HEME_CCD, smiles=None),
            input_builder.LigandInput(id=LIG_CHAIN, ccd=None, smiles=smiles)],
        pocket=input_builder.PocketConditioning(
            binder_chain_id=LIG_CHAIN, contacts=contacts))


def renumber_cif(text, shift, chain=PROT_CHAIN):
    """Shift the protein chain's author residue numbers to native numbering.

    ESMFold2 numbers the chain 1..len(PROT_SEQ); the construct starts at
    SEQ_FIRST, so native = output + (SEQ_FIRST - 1). Without this, Cys442
    comes back as 422 and every pocket definition downstream is off by 20.

    Edits the _atom_site loop in place rather than round-tripping through a
    gemmi Structure, which can drop metadata. Only auth_seq_id moves: gemmi
    reads that into seqid.num and the cif->pdb conversion follows it, while
    label_seq_id is the canonical 1..N entity index and must not change.
    Heme and ligand chains are untouched.
    """
    if not shift:
        return text, 0
    doc = gemmi.cif.read_string(text)
    blk = doc.sole_block()

    def _has(c):
        v = blk.find_values("_atom_site." + c)
        return v is not None and len(v) > 0

    seq_col = next((c for c in ("auth_seq_id", "label_seq_id") if _has(c)), None)
    asym_col = next((c for c in ("auth_asym_id", "label_asym_id")
                     if _has(c)), None)
    if seq_col is None or asym_col is None:
        raise RuntimeError("_atom_site has no usable seq/asym columns; "
                           "set RENUMBER_OUT = False and shift downstream")
    seen = set()
    for row in blk.find("_atom_site.", [asym_col, seq_col]):
        if gemmi.cif.as_string(row[0]).strip() != chain:
            continue
        v = gemmi.cif.as_string(row[1]).strip()
        if v in (".", "?", ""):
            continue
        row[1] = str(int(v) + shift)
        seen.add(int(v))
    return doc.as_string(), len(seen)


def lig_xyz_aligned(cif_path):
    """Protein CA coords and ligand heavy-atom coords from a predicted cif."""
    st = gemmi.read_structure(str(cif_path)); st.setup_entities()
    ca, lig = [], []
    for ch in st[0]:
        for r in ch:
            if ch.name == PROT_CHAIN:
                a = next((x for x in r if x.name.strip() == "CA"), None)
                if a:
                    ca.append(a.pos.tolist())
            elif ch.name == LIG_CHAIN:
                lig += [x.pos.tolist() for x in r if x.element.name != "H"]
    return np.array(ca), np.array(lig)


print("helpers defined: _f, fold_once, build_input, renumber_cif, "
      "lig_xyz_aligned")

In [ ]:
# ============================================================================
# 8 -- freeze the run definition, so a resume reproduces this exactly and a
#      later run can be compared against it.
# ============================================================================
RUN_DEF = RUN_DIR / "run_definition.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DEF.write_text(json.dumps({
    "run_name": RUN_NAME,
    "prot_seq": PROT_SEQ,
    "seq_first": SEQ_FIRST, "seq_last": SEQ_LAST, "seq_shift": SEQ_SHIFT,
    "full_seq_first_resnum": FULL_FIRST,
    "renumber_out": bool(RENUMBER_OUT),
    "ref_pdb": str(REF_PDB),
    "pocket_set": POCKET_SET,
    "pocket": {str(k): v for k, v in POCKET.items()},
    "verified_resnums": list(keep),
    "explicit_subpockets": bool(EXPLICIT_SUBPOCKETS),
    "n_subpockets_requested": N_SUBPOCKETS,
    "min_cluster": MIN_CLUSTER,
    "linkage": LINKAGE,
    "subpockets": {str(k): v for k, v in subpockets.items()},
    "set_resnums": {str(k): v for k, v in SET_RESNUMS.items()},
    "set_label": {str(k): v for k, v in SET_LABEL.items()},
    "contact_sets": {str(k): [[c, int(i)] for c, i in v]
                     for k, v in CONTACT_SETS.items()},
    "add_heme_contact": bool(ADD_HEME_CONTACT),
    "add_pair_sets": ADD_PAIR_SETS,
    "add_whole_pocket": bool(ADD_WHOLE_POCKET),
    "heme_ccd": HEME_CCD,
    "smiles": {c: SMILES[c] for c in CMPDS},
    "stereo_override": {k: (v if isinstance(v, list) else [v])
                        for k, v in STEREO_OVERRIDE.items()},
    "msa_sequences": list(FULL_MSA.sequences) if FULL_MSA is not None else None,
    "config": {"num_loops": CONFIG.num_loops,
               "num_sampling_steps": CONFIG.num_sampling_steps,
               "include_pae": CONFIG.include_pae,
               "include_pair_chains_iptm": CONFIG.include_pair_chains_iptm},
}, indent=1))
print(f"wrote {RUN_DEF}  ({RUN_DEF.stat().st_size/1e6:.1f} MB)")
print(f"  {len(CONTACT_SETS)} contact sets, {len(CMPDS)} compounds, "
      f"{N_FOLDS} folds")

In [ ]:
# ============================================================================
# 9 -- THE RUN. This is the only cell that spends tokens.
#
# seed   = contact set   (a binding hypothesis)
# sample = stereo variant (1 unless the compound has both enantiomers)
#
# Resumable: the score row is appended and flushed BEFORE the cif is written,
# so a crash never leaves a file without a row. Re-running skips what is done.
# ============================================================================
rows = (pd.read_csv(SCORES_CSV).to_dict("records")
        if SCORES_CSV.is_file() else [])
done = {r["Model"] for r in rows}
print(f"resuming with {len(rows)} existing score rows")

todo = [(c, s, vi) for c in CMPDS for s in SET_IDS
        for vi in range(1, len(SMILES[c]) + 1)
        if f"{c}_seed_{s}_sample_{vi}_model" not in done]
print(f"{len(todo)} of {N_FOLDS} folds to run\n")

t0, failed = time.time(), []
for i, (cmpd, sid, vi) in enumerate(todo, 1):
    stem = f"{cmpd}_seed_{sid}_sample_{vi}_model"
    cif  = CIF_DIR / f"{stem}.cif"
    try:
        r = fold_once(build_input(SMILES[cmpd][vi - 1],
                                  CONTACT_SETS[sid], FULL_MSA), CONFIG)
        text = r.complex.to_mmcif()
        n_renum = 0
        if RENUMBER_OUT:
            text, n_renum = renumber_cif(text, SEQ_SHIFT)
        row = {"Target_ID": cmpd, "Model": stem, "cmpd": cmpd,
               "id": OCNT_ID[cmpd], "seed": sid, "sample": vi,
               "contact_set": SET_LABEL[sid],
               "n_contacts": len(CONTACT_SETS[sid]),
               "heme_contact": ADD_HEME_CONTACT,
               "smiles": SMILES[cmpd][vi - 1],
               "n_variants": len(SMILES[cmpd]),
               "renumbered": n_renum,
               "ptm":   _f(getattr(r, "ptm", np.nan)),
               "plddt": _f(getattr(r, "plddt", np.nan)),
               "iptm":  _f(getattr(r, "iptm", np.nan)),
               "pair_chains_iptm": _f(getattr(r, "pair_chains_iptm", np.nan))}
        rows.append(row)
        pd.DataFrame(rows).to_csv(SCORES_CSV, index=False)   # flush first
        cif.write_text(text)                                 # then the file
        el = time.time() - t0
        print(f"  [{i}/{len(todo)}] {stem}  {SET_LABEL[sid]:8s}  "
              f"iptm {row['iptm']:.3f}  pair {row['pair_chains_iptm']:.3f}  "
              f"pLDDT {row['plddt']:.3f}  "
              f"({el/i:.0f}s/fold, ~{el/i*(len(todo)-i)/60:.0f} min left)",
              flush=True)
    except Exception as ex:
        failed.append((stem, str(ex)))
        print(f"  [{i}/{len(todo)}] [!] {stem} FAILED {ex}", flush=True)

SC = pd.DataFrame(rows)
SC.to_csv(SCORES_CSV, index=False)
SC.assign(iptm_AC=SC["iptm"], iptm_BC=SC["pair_chains_iptm"])[
    ["cmpd", "seed", "sample", "contact_set", "iptm_AC", "iptm_BC"]].to_csv(
        RUN_DIR / "harvest_iptm.csv", index=False)

print(f"\n{len(SC)}/{N_FOLDS} folds complete -> {CIF_DIR}")
print(f"iptm: {RUN_DIR / 'harvest_iptm.csv'}")
if len(SC):
    print("\nper-compound ipTM:")
    print(SC.groupby("cmpd")["iptm"].agg(["count","min","median","max"])
            .round(3).to_string())
    print("\nipTM by contact set (does one hypothesis dominate?):")
    print(SC.groupby("contact_set")["iptm"].agg(["count","mean","max"])
            .round(3).to_string())
for s, e in failed[:10]:
    print(f"  [!] {s}: {e}")

In [ ]:
# ============================================================================
# 10 -- RESUME in a fresh kernel: restore the frozen definition instead of
#       re-deriving it. Run cells 1 and 2, then this, then cell 9.
# ============================================================================
D = json.loads((RUN_DIR / "run_definition.json").read_text())

PROT_SEQ  = D["prot_seq"]
SEQ_FIRST, SEQ_LAST = D["seq_first"], D["seq_last"]
SEQ_SHIFT = D["seq_shift"]
RENUMBER_OUT = D["renumber_out"]
REF_PDB   = Path(D["ref_pdb"])
POCKET    = {int(k): v for k, v in D["pocket"].items()}
keep      = D["verified_resnums"]
subpockets = {int(k): v for k, v in D["subpockets"].items()}
CONTACT_SETS = {int(k): [(c, int(i)) for c, i in v]
                for k, v in D["contact_sets"].items()}
SET_LABEL   = {int(k): v for k, v in D["set_label"].items()}
SET_RESNUMS = {int(k): v for k, v in D["set_resnums"].items()}
SET_IDS     = sorted(CONTACT_SETS)
SMILES      = {k: list(v) for k, v in D["smiles"].items()}
CMPDS       = sorted(SMILES, key=lambda s: int(s.split("_")[1]))
ADD_HEME_CONTACT, HEME_CCD = D["add_heme_contact"], D["heme_ccd"]
FULL_MSA = (MSA.from_sequences(D["msa_sequences"])
            if D["msa_sequences"] else None)
CONFIG   = FoldingConfig(**D["config"])
N_FOLDS  = len(SET_IDS) * sum(len(v) for v in SMILES.values())

_ch  = pd.read_csv(CHALLENGE_CSV)
OCNT_ID = dict(zip(_ch["id2"].astype(str).str.strip(),
                   _ch["id"].astype(str).str.strip()))

def res_index(resnum, expect=None):
    i = resnum - SEQ_FIRST
    if not 0 <= i < len(PROT_SEQ):
        raise ValueError(f"residue {resnum} outside {SEQ_FIRST}-{SEQ_LAST}")
    if expect and PROT_SEQ[i] != expect:
        raise ValueError(f"residue {resnum} is {PROT_SEQ[i]!r}, want {expect!r}")
    return i

print(f"restored {RUN_NAME}: {len(PROT_SEQ)} residues "
      f"({SEQ_FIRST}-{SEQ_LAST}), {len(SET_IDS)} contact sets, "
      f"{len(CMPDS)} compounds, {N_FOLDS} folds")
print(f"  MSA depth {len(D['msa_sequences']) if D['msa_sequences'] else 0}")
print("  now run cell 7 (helpers) and cell 9 (the run)")

In [ ]:
# ============================================================================
# 11 -- QC the output, and ask the question the leaderboard analysis raised:
#       do the contact sets actually span distinct binding modes, or is the
#       generator collapsing to one answer regardless of conditioning?
#
#   wide spread  -> selection is the bottleneck; a better selector pays
#   tight spread -> generation is the bottleneck; conditioning is not working
# ============================================================================
cifs = sorted(CIF_DIR.glob("*_model.cif"))
print(f"{len(cifs)} predictions in {CIF_DIR}")

# --- structural QC on one file --------------------------------------------
if cifs:
    p = cifs[0]
    st = gemmi.read_structure(str(p)); st.setup_entities()
    print(f"\n{p.name}: {len(st[0])} chains")
    for ch in st[0]:
        nums = [r.seqid.num for r in ch]
        het = sorted({r.name for r in ch if r.het_flag == "H"})
        print(f"  {ch.name!r}: {len(ch):4d} res "
              f"{min(nums) if nums else '-'}-{max(nums) if nums else '-'}  "
              f"het={het[:3]}")
    prot_nums = [r.seqid.num for ch in st[0] if ch.name == PROT_CHAIN for r in ch]
    if prot_nums:
        lo, hi = min(prot_nums), max(prot_nums)
        ok = (lo, hi) == (SEQ_FIRST, SEQ_LAST)
        print(f"\nprotein numbered {lo}-{hi}; expected {SEQ_FIRST}-{SEQ_LAST}"
              f"  {'OK' if ok else '<-- renumbering did not apply'}")
        _c = {r.seqid.num: r.name for ch in st[0] if ch.name == PROT_CHAIN
              for r in ch}.get(CYS_RESNUM)
        print(f"residue {CYS_RESNUM} is {_c} (expect CYS)")
    names = {r.name.strip().upper() for ch in st[0] for r in ch}
    print(f"heme present: {'HEM' in names}")

# --- ligand spread across contact sets, per compound ----------------------
recs = []
for c in sorted({p.stem.split("_seed_")[0] for p in cifs},
                key=lambda s: int(s.split("_")[1])):
    ps = sorted(p for p in cifs if p.stem.startswith(c + "_seed_"))
    xs = {}
    for p in ps:
        try:
            cav, lig = lig_xyz_aligned(p)
            if len(cav) and len(lig):
                xs[p.stem] = (cav, lig)
        except Exception:
            pass
    ks = sorted(xs)
    if len(ks) < 2:
        continue
    d = []
    for a in range(len(ks)):
        for b in range(a + 1, len(ks)):
            c1, l1 = xs[ks[a]]; c2, l2 = xs[ks[b]]
            if c1.shape != c2.shape or l1.shape != l2.shape:
                continue
            m1, m2 = c1.mean(0), c2.mean(0)
            U, _S, Vt = np.linalg.svd((c2 - m2).T @ (c1 - m1))
            Rot = Vt.T @ np.diag([1, 1, np.sign(np.linalg.det(Vt.T @ U.T))]) @ U.T
            l2a = (Rot @ (l2 - m2).T).T + m1
            d.append(float(np.sqrt(((l1 - l2a) ** 2).sum(1).mean())))
    if d:
        recs.append({"cmpd": c, "n_poses": len(ks), "min": min(d),
                     "median": float(np.median(d)), "max": max(d)})

if recs:
    DIV = pd.DataFrame(recs)
    print("\n--- pairwise ligand RMSD between contact sets (A) ---")
    print(DIV.round(2).to_string(index=False))
    print(f"\nacross all compounds: median spread "
          f"{DIV['median'].median():.2f} A, "
          f"{int((DIV['median'] < 1.0).sum())}/{len(DIV)} compounds collapsed "
          f"below 1.0 A")
    print("A compound whose poses all agree gains nothing from a better "
          "selector -- its conditioning is not producing alternatives.")

In [ ]:
# ============================================================================
# 12 -- DIAGNOSTIC (not part of the run). Resolve numbering by hand if cell 4
#       raises: shows which offsets reproduce the pocket residue identities,
#       for both the template sequence and REF_PDB's own numbering.
# ============================================================================
print(f"template sequence: {len(FULL_SEQ)} residues")
for frac, hit, tot, off in derive_first_resnum(FULL_SEQ, POCKET)[1]:
    print(f"  first_resnum = {off:4d}   {hit:2d}/{tot:2d} match")

_st = gemmi.read_structure(str(REF_PDB)); _st.setup_entities()
_lk = {r.seqid.num: r.name.strip().upper()
       for ch in _st[0] if ch.name == PROT_CHAIN for r in ch}
print(f"\nREF_PDB chain {PROT_CHAIN}: {len(_lk)} residues, "
      f"numbered {min(_lk)}-{max(_lk)}")
rs = []
for sh in range(-120, 121):
    hit = tot = 0
    for n, nm3 in POCKET.items():
        got = _lk.get(n + sh)
        if got:
            tot += 1
            hit += got == nm3
    if tot:
        rs.append((hit, tot, sh))
rs.sort(reverse=True)
print("best REF_PDB shift (structure_resnum = pocket_resnum + shift):")
for hit, tot, sh in rs[:4]:
    print(f"  shift = {sh:4d}   {hit:2d}/{tot:2d}"
          + ("   <-- all match" if hit == tot == len(POCKET) else ""))

print(f"\nsliced sequence, numbered from {SEQ_FIRST}; pocket in lower case:")
marked = "".join(ch.lower() if (i + SEQ_FIRST) in POCKET else ch
                 for i, ch in enumerate(PROT_SEQ))
for s in range(0, len(PROT_SEQ), 60):
    print(f"  {s + SEQ_FIRST:>5d}  {marked[s:s + 60]}")